# **Semantic Web & Linked Open Data with Python**
## บทที่ 05 — Knowledge Management

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch05/ch05_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://owl-rl.readthedocs.io/
*   https://raw.githubusercontent.com/Rathachai/semantic-web/main/code/ch05/university_tbox.ttl

📖 เอกสารประกอบบทนี้: [`05-knowledge-management/`](https://github.com/Rathachai/semantic-web/tree/main/05-knowledge-management)

---

### Setup

In [1]:
!pip install -q rdflib owlrl

In [2]:
import rdflib, owlrl
from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL

UNI = Namespace('http://example.org/university#')

## To Build a Knowledge Base: TBox + RBox + ABox

**TBox** = นิยาม class, **RBox** = นิยาม property/role, **ABox** = ข้อเท็จจริง

📄 เวอร์ชันเต็ม: [`university_tbox.ttl`](university_tbox.ttl) + [`university_abox.ttl`](university_abox.ttl)

In [3]:
tbox_ttl = """
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix uni:  <http://example.org/university#> .

uni:Person a owl:Class .
uni:Student a owl:Class ; rdfs:subClassOf uni:Person .
uni:Course a owl:Class .
uni:GraduateCourse a owl:Class ; rdfs:subClassOf uni:Course .

uni:enrolledIn a rdf:Property ; rdfs:domain uni:Student ; rdfs:range uni:Course .
uni:teaches a rdf:Property ; rdfs:domain uni:Person ; rdfs:range uni:Course .
uni:instructorOf a rdf:Property ; rdfs:subPropertyOf uni:teaches .
"""

abox_ttl = """
@prefix uni: <http://example.org/university#> .

uni:alice a uni:Student ; uni:enrolledIn uni:cs501 .
uni:cs501 a uni:GraduateCourse .
uni:profSmith uni:instructorOf uni:cs501 .
uni:dave a uni:Student .
"""

g = Graph()
g.parse(data=tbox_ttl, format="turtle")
g.parse(data=abox_ttl, format="turtle")
print(f"{len(g)} triples")

19 triples


## Open World Assumption: dave ไม่มี enrolledIn

`dave` เป็น `Student` แต่ไม่เคยประกาศ `enrolledIn` เลย — ลอง query ดู

In [4]:
res = g.query("SELECT ?course WHERE { uni:dave uni:enrolledIn ?course . }",
               initNs={"uni": UNI})
print("ผลลัพธ์:", list(res))
print("\n⚠️ ผลว่างเปล่า ≠ 'dave ไม่ได้ลงทะเบียนวิชาใดเลย' -- แปลว่า 'ไม่รู้' ภายใต้ OWA")

ผลลัพธ์: []

⚠️ ผลว่างเปล่า ≠ 'dave ไม่ได้ลงทะเบียนวิชาใดเลย' -- แปลว่า 'ไม่รู้' ภายใต้ OWA


## Instance Retrieval ก่อน reasoning

"ใครเป็น Person บ้าง?" -- ก่อนรัน reasoner จะเห็นแค่คนที่ประกาศ `a uni:Person` ตรง ๆ (ไม่มีเลย!)

In [5]:
res = g.query("SELECT ?p WHERE { ?p a uni:Person . }", initNs={"uni": UNI})
print(list(res))

[]


## รัน owlrl (OWL 2 RL) — ไม่ต้องใช้ Java

เหมาะกับข้อมูลขนาดใหญ่ที่ต้องการความเร็ว แลกกับพลังการอนุมานที่จำกัดกว่า OWL DL เต็มรูปแบบ

In [6]:
owlrl.DeductiveClosure(owlrl.OWLRL_Extension).expand(g)
print(f'{len(g)} triples หลัง reasoning')

256 triples หลัง reasoning


## Instance Retrieval หลัง reasoning

ตอนนี้ `alice`, `dave`, `profSmith` ควรปรากฏเป็น `Person` เพราะ `Student ⊑ Person`

In [7]:
res = g.query("SELECT ?p WHERE { ?p a uni:Person . }", initNs={"uni": UNI})
for row in res:
    print(row.p)

http://example.org/university#alice
http://example.org/university#dave
http://example.org/university#profSmith


## Role Chain: instructorOf ⊑ teaches

`profSmith` ไม่เคยถูกประกาศว่า `teaches` โดยตรง — แต่ `instructorOf` เป็น subProperty

In [8]:
res = g.query("SELECT ?course WHERE { uni:profSmith uni:teaches ?course . }",
               initNs={"uni": UNI})
print(list(res))
print("\n👀 subPropertyOf ทำให้ instructorOf(profSmith, cs501) => teaches(profSmith, cs501) ด้วย")

[(rdflib.term.URIRef('http://example.org/university#cs501'),)]

👀 subPropertyOf ทำให้ instructorOf(profSmith, cs501) => teaches(profSmith, cs501) ด้วย


## 🎯 สรุป

- **TBox / RBox / ABox** คือสามส่วนของ knowledge base
- **OWA**: ไม่มี triple ≠ เป็นเท็จ — dave อาจลงทะเบียนวิชาอยู่ก็ได้ แค่เรายังไม่รู้
- `owlrl.OWLRL_Extension` ทำงานได้โดยไม่ต้องใช้ Java เหมาะกับข้อมูลขนาดใหญ่
- **subPropertyOf** ทำให้ property ที่เจาะจงกว่า "ยก" ความจริงขึ้นไปยัง property ทั่วไปได้

ลองเพิ่ม `uni:erin a uni:GraduateCourse` เข้าไปเป็น subject แล้วดูว่า reasoner อนุมานอะไรเพิ่ม!

---

## จบบทที่ 05 🎉

ไปต่อบทถัดไป: **บทที่ 06 — SPARQL**

📚 กลับหน้าคอร์ส: [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*